# Tarea 3 · Clasificación (regresión logística)

Enunciado: [homework.md](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md)

- **Dataset:** `course_lead_scoring_2026.csv` (no se versiona; descárgalo en esta carpeta):
  `curl -LO https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv`
- **Objetivo:** predecir `converted` (¿el cliente se dio de alta en la plataforma?).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## 1. Carga y preparación del dataset

In [ ]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [ ]:
# ¿Hay valores nulos? ¿En qué columnas?
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [ ]:
# Separamos categóricas (texto) y numéricas; 'converted' es la variable objetivo
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

# Imputación según el enunciado: 'NA' en categóricas y 0.0 en numéricas
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
df.isnull().sum().sum()

np.int64(0)

**Conclusión:** hay nulos en las 4 categóricas y en `annual_income` y `lead_score`. Tras imputar no queda ninguno.
Rellenar con `'NA'` convierte "no sabemos" en una categoría más, que el modelo puede aprender.

In [ ]:
# Balance de clases: si estuviera muy desequilibrado, la accuracy engañaría
df['converted'].value_counts(normalize=True).round(3)

converted
1    0.577
0    0.423
Name: proportion, dtype: float64

## Question 1: Moda de `industry`

In [ ]:
df['industry'].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

**Respuesta Q1:** `technology` (1173 observaciones). Aunque `NA` aparece 240 veces, no es la más frecuente.

## Question 2: Matriz de correlación

In [ ]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [ ]:
pairs = [('interaction_count', 'lead_score'),
         ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'),
         ('annual_income', 'interaction_count')]
for a, b in pairs:
    print(f'{a} - {b}: {corr.loc[a, b]:.3f}')

interaction_count - lead_score: 0.916
number_of_courses_viewed - lead_score: 0.757
number_of_courses_viewed - interaction_count: 0.722
annual_income - interaction_count: 0.123


**Respuesta Q2:** `interaction_count` y `lead_score` (0.916).

## División train / validation / test (60/20/20)

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train['converted'].values
y_val = df_val['converted'].values

# Quitamos la variable objetivo de las features (evita data leakage)
df_train = df_train.drop(columns='converted')
df_val = df_val.drop(columns='converted')
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3: Mutual information (solo con train)

In [ ]:
mi = {c: round(mutual_info_score(y_train, df_train[c]), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

**Respuesta Q3:** `lead_source` (0.03). Es la categórica que más información aporta sobre la conversión; `industry` y `location` casi nada.

## Question 4: Regresión logística con one-hot

In [ ]:
features = categorical + numerical

def train_and_score(feature_list, C=1.0):
    """Entrena con one-hot (DictVectorizer) y devuelve la accuracy en validación."""
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feature_list].to_dict(orient='records'))
    X_val = dv.transform(df_val[feature_list].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

base_accuracy = train_and_score(features)
base_accuracy, round(base_accuracy, 2)

(0.645, 0.65)

**Respuesta Q4:** 0.65 (exactamente 0.645).

## Question 5: Eliminación de features

In [ ]:
diffs = {}
for f in features:
    acc = train_and_score([x for x in features if x != f])
    diffs[f] = base_accuracy - acc
pd.Series(diffs).sort_values()

annual_income              -0.079
industry                    0.000
employment_status           0.000
location                    0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [ ]:
# Solo las tres opciones del enunciado
{f: round(diffs[f], 3) for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']}

{'lead_source': 0.003,
 'number_of_courses_viewed': 0.002,
 'interaction_count': 0.044}

**Respuesta Q5:** `number_of_courses_viewed` (diferencia 0.002, la menor de las tres opciones).

**Observación:** quitar `annual_income` *mejora* la accuracy (0.645 → 0.724). No es que el ingreso no importe: está en una escala enorme (decenas de miles) sin normalizar, y domina a las demás features en el ajuste. Se arreglaría escalando (lo veremos más adelante).

## Question 6: Regularización (parámetro C)

In [ ]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


**Respuesta Q6:** `C = 0.001` (0.645). En `LogisticRegression`, **C es la inversa** de la fuerza de regularización: C muy pequeño = penalización muy fuerte → pesos casi nulos → el modelo apenas aprende (0.598 ≈ predecir siempre la clase mayoritaria en validación).

## Resumen de respuestas

| Q | Respuesta |
|---|---|
| 1 | `technology` |
| 2 | `interaction_count` y `lead_score` |
| 3 | `lead_source` |
| 4 | 0.65 |
| 5 | `number_of_courses_viewed` |
| 6 | 0.001 |